In [ ]:
from retrievers.vector_retriever import VectorRetriever
from retrievers.fulltext_retriever import FullTextRetriever
from retrievers.hybrid_retriever import HybridRetriever
from retrievers.text2cyper import Text2CypherRetriever
from graph.neo4j_manager import Neo4jManager
from agents.multi_agent_system import MultiAgentSystem
from agents.question_descomposer import QuestionDecomposer


In [ ]:
neo4j = Neo4jManager()


In [ ]:
query = "Who is Momotaro?"

extra_match = """
OPTIONAL MATCH (node)-[:HAS_ROLE]->(r:Role)
"""

return_projection={
	"name": "node.name",
	"description": "node.description",
	"role": "r.name"
}
retriever = VectorRetriever(neo4j, "agent_embeddings", return_projection, extra_match)
results = retriever.retrieve(query)

print(f"Vector search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]} [{r["role"]}]: {r["description"]}\n")


In [ ]:
query = "millet dumpling"
return_fields = {
    "description": "node.description",
    "name": "node.name"
}
retriever = FullTextRetriever(neo4j, "event_fulltext", "Event", "description", return_fields)
results = retriever.retrieve(query)

print(f"Fulltext search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
query = "How did Momotaro convince the animals to join him on his journey?"
return_fields = {
    "description": "node.description",
    "name": "node.name"
}
retriever = HybridRetriever(neo4j, "agent_embeddings", "agent_fulltext", "Agent", "description", vector_weight=0.5, return_projection=return_fields, include_score=True)
results = retriever.retrieve(query, 10)

print(f"Hybrid search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
text2cypher = Text2CypherRetriever(neo4j)

text2cypher.add_few_shot_example(
    "What happens in Momotaro, in order, and which characters are involved in each event?",
    """
    MATCH (f:Folktale {url: "https://fairytalez.com/momotaro/"})-[:HAS_EVENT]->(e:Event)
    OPTIONAL MATCH (e)-[:HAS_AGENT]->(a:Agent)
    RETURN e.order AS order,
           e.name AS event,
           collect(a.name) AS agents
    ORDER BY order
    """
)

text2cypher.add_few_shot_example(
    "Which characters are most important in the story based on how many events they appear in?",
    """
    MATCH (a:Agent)
    OPTIONAL MATCH (a)<-[:HAS_AGENT]-(e:Event)
    RETURN a.name AS agent,
           count(DISTINCT e) AS event_count
    ORDER BY event_count DESC
    LIMIT 10
    """
)

text2cypher.add_few_shot_example(
    "How many folktales come from Japan?",
    """
    MATCH (f:Folktale)-[:FROM_NATION]->(n:Nation {name: "Japan"})
    RETURN count(f) AS total_folktales
    """
)

text2cypher.add_few_shot_example(
    "Who are Momotaro's friends?",
    """
    MATCH (m:Agent {name: "Momotaro"})-[r:RELATIONSHIP]->(a:Agent)
    WHERE r.type = "friend"
    RETURN a.name AS friend
    """
)

text2cypher.add_few_shot_example(
    "What happens after the specific event where Momotaro gives the dumpling to the dog?",
    """
    MATCH (e:Event {name: "Momotaro gives dog a dumpling."})-[:POST_EVENT]->(next:Event)
    RETURN e.name AS current_event,
           next.name AS next_event
    """
)


In [ ]:
rag = MultiAgentSystem(neo4j)


In [ ]:
result = rag.answer("How did Momotaro convince the animals to join him on his journey?", "test1")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)


In [ ]:
result = rag.answer("How many characters have the role of helper?", "test4")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



In [ ]:
result = rag.answer("Where do the ogres live in the folkt of Momotaro?", "test")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



In [ ]:
result = rag.answer("What is the weather", "duh")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)


In [ ]:
neo4j.close()
